In [1]:
!pip install -q ultralytics fastapi uvicorn pyngrok pillow nest_asyncio python-multipart rasterio mercantile


In [ ]:
!rm -f ngrok ngrok.zip ngrok-stable-linux-amd64.tgz
!wget -O ngrok-stable-linux-amd64.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-stable-linux-amd64.tgz
!chmod +x ngrok
!./ngrok version
!./ngrok authtoken  "YOUR TOKEN HERE"


In [ ]:
import os, io, json, requests, nest_asyncio, uvicorn, mercantile
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok
from PIL import Image
import numpy as np
from ultralytics import YOLO

nest_asyncio.apply()

# ==========================================================
# 1. Basemap Configuration (Multi-Basemap Support)
# ==========================================================
BASEMAPS = {
    "nlsc_uav": {
        "name": "Taiwan NLSC UAV Imagery",
        "url": "https://wmts.nlsc.gov.tw/wmts/UAV111A03/default/GoogleMapsCompatible/{z}/{y}/{x}.png",
        "max_zoom": 20,
        "detect_zoom": 20,
        "attribution": "Taiwan NLSC UAV WMTS"
    },
    "nlsc_photo2": {
        "name": "Taiwan NLSC Ortho Photo",
        "url": "https://wmts.nlsc.gov.tw/wmts/PHOTO2/default/GoogleMapsCompatible/{z}/{y}/{x}",
        "max_zoom": 20,
        "detect_zoom": 19,
        "attribution": "Taiwan NLSC PHOTO2"
    },
    "nlsc_emap": {
        "name": "Taiwan NLSC e-Map",
        "url": "https://wmts.nlsc.gov.tw/wmts/EMAP/default/GoogleMapsCompatible/{z}/{y}/{x}",
        "max_zoom": 20,
        "detect_zoom": 19,
        "attribution": "Taiwan NLSC EMAP"
    },
    "esri_satellite": {
        "name": "Esri World Imagery",
        "url": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        "max_zoom": 19,
        "detect_zoom": 19,
        "attribution": "Tiles &copy; Esri"
    },
    "google_satellite": {
        "name": "Google Satellite",
        "url": "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        "max_zoom": 20,
        "detect_zoom": 20,
        "attribution": "Google"
    },
    "osm": {
        "name": "OpenStreetMap",
        "url": "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        "max_zoom": 19,
        "detect_zoom": 19,
        "attribution": "&copy; OpenStreetMap contributors"
    }
}

# ==========================================================
# 2. Model Registry (Multi-Model & Multi-Class Support)
#    Supports task='obb', 'detect', or 'segment'
#    Add custom building / road-marking weights here!
# ==========================================================
MODEL_REGISTRY = {
    "vehicles_obb": {
        "name": "Vehicles Only (YOLOv11x-OBB)",
        "weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": ["small vehicle", "large vehicle"],
        "description": "Detects small and large vehicles with oriented bounding boxes."
    },
    "dota_multiclass_obb": {
        "name": "Multi-Class Aerial (DOTAv1 YOLOv11x-OBB)",
        "weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": None,  # All 15 DOTAv1 classes (vehicles, ships, planes, bridges, etc.)
        "description": "Detects all 15 aerial object classes from DOTAv1."
    },
    "general_detect": {
        "name": "General Objects (YOLOv11x Detect)",
        "weights": "yolo11x.pt",
        "task": "detect",
        "allowed_classes": None,
        "description": "Standard horizontal bounding box detection (auto-downloads weights)."
    },
    "building_seg": {
        "name": "Buildings (Custom YOLOv11-Seg)",
        "weights": "/content/building-seg.pt",
        "task": "segment",
        "allowed_classes": None,
        "description": "Custom building footprint segmentation model (upload weights to /content/building-seg.pt)."
    },
    "road_marking_obb": {
        "name": "Road Markings (Custom YOLOv11-OBB)",
        "weights": "/content/road-marking-obb.pt",
        "task": "obb",
        "allowed_classes": None,
        "description": "Custom road-marking OBB model (upload weights to /content/road-marking-obb.pt)."
    }
}

DEFAULT_BASEMAP = "nlsc_uav"
DEFAULT_MODEL = "vehicles_obb"
IOU_THRESH = 0.5
TILE_SIZE = 256
HTTP_HEADERS = {"User-Agent": "UAV-Detection-App/2.0"}

# Cache loaded YOLO models by weights path so shared weights load only once
LOADED_MODELS = {}

def get_model(model_id: str):
    if model_id not in MODEL_REGISTRY:
        raise HTTPException(status_code=400, detail=f"Unknown model_id: {model_id}")
    cfg = MODEL_REGISTRY[model_id]
    weights = cfg["weights"]
    if weights not in LOADED_MODELS:
        # If a custom /content/*.pt path is configured, verify it exists before loading
        if weights.startswith("/") and not os.path.exists(weights):
            raise HTTPException(
                status_code=404,
                detail=f"Model weights not found at '{weights}'. Please upload the weights file or choose another model."
            )
        print(f"Loading model weights: {weights} ...")
        LOADED_MODELS[weights] = YOLO(weights)
        print("Loaded classes:", LOADED_MODELS[weights].names)
    return LOADED_MODELS[weights], cfg

# Preload default model if weights exist
if os.path.exists(MODEL_REGISTRY[DEFAULT_MODEL]["weights"]):
    get_model(DEFAULT_MODEL)
else:
    print(f"Note: Default weights '{MODEL_REGISTRY[DEFAULT_MODEL]['weights']}' not found yet; models will load on demand.")

app = FastAPI(title="UAV Multi-Model & Multi-Basemap Detection API")

class DetectRequest(BaseModel):
    sw_lat: float
    sw_lon: float
    ne_lat: float
    ne_lon: float
    basemap: str = DEFAULT_BASEMAP
    model_id: str = DEFAULT_MODEL
    conf: float = Field(default=0.3, ge=0.05, le=0.95)

def xy_to_lonlat(x, y, tile, img_w=TILE_SIZE, img_h=TILE_SIZE):
    bounds = mercantile.bounds(tile)
    lon = bounds.west + (x / float(img_w)) * (bounds.east - bounds.west)
    lat = bounds.north - (y / float(img_h)) * (bounds.north - bounds.south)
    return lon, lat

def download_tile(tile, url_template: str, zoom: int):
    url = url_template.format(z=zoom, x=tile.x, y=tile.y)
    r = requests.get(url, headers=HTTP_HEADERS, timeout=15)
    r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    return np.array(img), tile

def extract_polygons(results, model, task: str):
    """Extract (polygon_xy_points, label, score) tuples for obb, segment, or detect tasks."""
    detections = []
    if task == "obb" and getattr(results, "obb", None) is not None and len(results.obb) > 0:
        for poly, c, s in zip(results.obb.xyxyxyxy.tolist(), results.obb.cls.tolist(), results.obb.conf.tolist()):
            detections.append((poly, model.names[int(c)].lower(), float(s)))
    elif task == "segment" and getattr(results, "masks", None) is not None and results.masks is not None:
        for poly, c, s in zip(results.masks.xy, results.boxes.cls.tolist(), results.boxes.conf.tolist()):
            if len(poly) >= 3:
                detections.append((poly.tolist(), model.names[int(c)].lower(), float(s)))
    elif getattr(results, "boxes", None) is not None and len(results.boxes) > 0:
        for box, c, s in zip(results.boxes.xyxy.tolist(), results.boxes.cls.tolist(), results.boxes.conf.tolist()):
            x1, y1, x2, y2 = box
            poly = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
            detections.append((poly, model.names[int(c)].lower(), float(s)))
    return detections

@app.get("/config")
async def get_config():
    models_info = {}
    for mid, cfg in MODEL_REGISTRY.items():
        weights = cfg["weights"]
        available = (not weights.startswith("/")) or os.path.exists(weights)
        models_info[mid] = {
            "name": cfg["name"],
            "task": cfg["task"],
            "description": cfg["description"],
            "available": available
        }
    return {
        "basemaps": BASEMAPS,
        "models": models_info,
        "default_basemap": DEFAULT_BASEMAP,
        "default_model": DEFAULT_MODEL
    }

@app.post("/detect")
async def detect(req: DetectRequest):
    if req.basemap not in BASEMAPS:
        raise HTTPException(status_code=400, detail=f"Unknown basemap: {req.basemap}")

    basemap_cfg = BASEMAPS[req.basemap]
    zoom = basemap_cfg["detect_zoom"]
    model, model_cfg = get_model(req.model_id)
    task = model_cfg["task"]
    allowed_classes = model_cfg.get("allowed_classes")

    tiles = list(mercantile.tiles(req.sw_lon, req.sw_lat, req.ne_lon, req.ne_lat, zoom))
    features = []

    for t in tiles:
        try:
            tile_np, _ = download_tile(t, basemap_cfg["url"], zoom)
        except Exception as e:
            print(f"Skipping tile {t}: {e}")
            continue

        img_h, img_w = tile_np.shape[:2]
        results = model.predict(
            tile_np,
            conf=req.conf,
            iou=IOU_THRESH,
            imgsz=max(img_h, img_w),
            task=task,
            verbose=False
        )[0]

        for poly, label, score in extract_polygons(results, model, task):
            if allowed_classes is not None and label not in allowed_classes:
                continue

            coords_wgs84 = [xy_to_lonlat(x, y, t, img_w, img_h) for x, y in poly]
            lats = [lat for lon, lat in coords_wgs84]
            lons = [lon for lon, lat in coords_wgs84]

            # Filter detections inside user rectangle
            if min(lats) >= req.sw_lat and max(lats) <= req.ne_lat and \
               min(lons) >= req.sw_lon and max(lons) <= req.ne_lon:
                ring = [[lon, lat] for lon, lat in coords_wgs84]
                if ring[0] != ring[-1]:
                    ring.append(ring[0])  # Close GeoJSON polygon ring
                features.append({
                    "type": "Feature",
                    "properties": {
                        "label": label,
                        "score": round(score, 4),
                        "model": req.model_id,
                        "basemap": req.basemap
                    },
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [ring]
                    }
                })

    return {"type": "FeatureCollection", "features": features}

@app.get("/", response_class=HTMLResponse)
async def index():
    basemaps_json = json.dumps(BASEMAPS)
    models_meta = {
        mid: {
            "name": cfg["name"],
            "task": cfg["task"],
            "description": cfg["description"],
            "available": (not cfg["weights"].startswith("/")) or os.path.exists(cfg["weights"])
        }
        for mid, cfg in MODEL_REGISTRY.items()
    }
    models_json = json.dumps(models_meta)

    html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8"/>
<title>UAV Object Detection (Multi-Basemap & Multi-Model)</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>
<style>
html, body, #map {{ height: 100%; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
.panel {{ position: absolute; top: 10px; right: 10px; z-index: 1000; background: rgba(255,255,255,0.95); padding: 12px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.25); width: 250px; }}
.panel h4 {{ margin: 0 0 8px 0; font-size: 14px; color: #1e293b; }}
.field {{ margin-bottom: 8px; }}
.field label {{ display: block; font-size: 12px; font-weight: 600; color: #475569; margin-bottom: 3px; }}
.field select, .field input[type=range] {{ width: 100%; box-sizing: border-box; padding: 4px 6px; border-radius: 4px; border: 1px solid #cbd5e1; font-size: 12px; }}
.btn {{ display: block; width: 100%; text-align: center; background: #2563eb; color: #fff; padding: 7px; margin-top: 6px; cursor: pointer; border: none; border-radius: 5px; font-size: 13px; font-weight: 600; }}
.btn:hover {{ background: #1d4ed8; }}
.btn.secondary {{ background: #64748b; }}
.btn.secondary:hover {{ background: #475569; }}
.btn.success {{ background: #059669; }}
.btn.success:hover {{ background: #047857; }}
#status {{ margin-top: 8px; font-size: 12px; color: #334155; }}
#legend {{ margin-top: 8px; font-size: 11px; max-height: 150px; overflow-y: auto; border-top: 1px solid #e2e8f0; padding-top: 6px; }}
.legend-item {{ display: flex; align-items: center; gap: 6px; margin-bottom: 3px; }}
.swatch {{ width: 10px; height: 10px; border-radius: 2px; display: inline-block; }}
</style>
</head>
<body>
<div id="map"></div>
<div class="panel">
  <h4>UAV Object Detection</h4>
  <div class="field">
    <label for="basemapSelect">Basemap</label>
    <select id="basemapSelect"></select>
  </div>
  <div class="field">
    <label for="modelSelect">Detection Model</label>
    <select id="modelSelect"></select>
  </div>
  <div class="field">
    <label for="confSlider">Confidence: <span id="confVal">0.30</span></label>
    <input id="confSlider" type="range" min="0.10" max="0.90" step="0.05" value="0.30"/>
  </div>
  <button class="btn" id="btnDetect">Detect Objects</button>
  <button class="btn success" id="btnDownload">Download GeoJSON</button>
  <button class="btn secondary" id="btnClear">Clear Detections</button>
  <div id="status">Draw a rectangle on the map to begin.</div>
  <div id="legend"></div>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<script>
const BASEMAPS = {basemaps_json};
const MODELS = {models_json};
const DEFAULT_BASEMAP = '{DEFAULT_BASEMAP}';
const DEFAULT_MODEL = '{DEFAULT_MODEL}';

const CLASS_COLORS = {{
    'small vehicle': '#ef4444',
    'large vehicle': '#2563eb',
    'plane': '#f59e0b',
    'ship': '#06b6d4',
    'storage tank': '#8b5cf6',
    'bridge': '#ec4899',
    'harbor': '#14b8a6',
    'building': '#f97316',
    'road marking': '#eab308'
}};
const PALETTE = ['#ef4444','#2563eb','#10b981','#f59e0b','#8b5cf6','#ec4899','#06b6d4','#f97316','#84cc16','#6366f1'];

function getClassColor(label) {{
    if (CLASS_COLORS[label]) return CLASS_COLORS[label];
    let hash = 0;
    for (let i = 0; i < label.length; i++) hash = (hash * 31 + label.charCodeAt(i)) | 0;
    return PALETTE[Math.abs(hash) % PALETTE.length];
}}

const map = L.map('map').setView([25.0330, 121.5054], 15);
let currentTileLayer = null;

function setBasemap(key) {{
    const cfg = BASEMAPS[key];
    if (!cfg) return;
    if (currentTileLayer) map.removeLayer(currentTileLayer);
    currentTileLayer = L.tileLayer(cfg.url, {{ maxZoom: cfg.max_zoom, attribution: cfg.attribution }}).addTo(map);
}}

const basemapSelect = document.getElementById('basemapSelect');
Object.entries(BASEMAPS).forEach(([k, cfg]) => {{
    const opt = document.createElement('option');
    opt.value = k;
    opt.textContent = cfg.name;
    if (k === DEFAULT_BASEMAP) opt.selected = true;
    basemapSelect.appendChild(opt);
}});
basemapSelect.onchange = () => setBasemap(basemapSelect.value);
setBasemap(DEFAULT_BASEMAP);

const modelSelect = document.getElementById('modelSelect');
Object.entries(MODELS).forEach(([k, cfg]) => {{
    const opt = document.createElement('option');
    opt.value = k;
    opt.textContent = cfg.available ? cfg.name : `${{cfg.name}} (upload weights)`;
    if (k === DEFAULT_MODEL) opt.selected = true;
    modelSelect.appendChild(opt);
}});

const confSlider = document.getElementById('confSlider');
const confVal = document.getElementById('confVal');
confSlider.oninput = () => {{ confVal.textContent = parseFloat(confSlider.value).toFixed(2); }};

const drawnItems = new L.FeatureGroup().addTo(map);
map.addControl(new L.Control.Draw({{
    draw: {{ rectangle: true, polygon: false, polyline: false, circle: false, marker: false, circlemarker: false }},
    edit: {{ featureGroup: drawnItems }}
}}));
map.on(L.Draw.Event.CREATED, e => {{
    drawnItems.clearLayers();
    drawnItems.addLayer(e.layer);
}});

const detectionLayer = L.layerGroup().addTo(map);
const statusEl = document.getElementById('status');
const legendEl = document.getElementById('legend');
let lastGeoJSON = null;

function updateLegend(counts) {{
    const entries = Object.entries(counts);
    if (entries.length === 0) {{ legendEl.innerHTML = ''; return; }}
    legendEl.innerHTML = entries
        .sort((a, b) => b[1] - a[1])
        .map(([label, cnt]) => `<div class="legend-item"><span class="swatch" style="background:${{getClassColor(label)}}"></span>${{label}} (${{cnt}})</div>`)
        .join('');
}}

document.getElementById('btnDetect').onclick = async () => {{
    if (drawnItems.getLayers().length === 0) {{ alert('Draw a rectangle first!'); return; }}
    const rect = drawnItems.getLayers()[0].getBounds();
    const payload = {{
        sw_lat: rect.getSouthWest().lat,
        sw_lon: rect.getSouthWest().lng,
        ne_lat: rect.getNorthEast().lat,
        ne_lon: rect.getNorthEast().lng,
        basemap: basemapSelect.value,
        model_id: modelSelect.value,
        conf: parseFloat(confSlider.value)
    }};
    detectionLayer.clearLayers();
    updateLegend({{}});
    statusEl.textContent = 'Running detection...';

    try {{
        const res = await fetch('/detect', {{
            method: 'POST',
            headers: {{ 'Content-Type': 'application/json' }},
            body: JSON.stringify(payload)
        }});
        const data = await res.json();
        if (!res.ok) {{
            statusEl.textContent = 'Error: ' + (data.detail || res.statusText);
            return;
        }}
        lastGeoJSON = data;
        const counts = {{}};
        data.features.forEach(f => {{
            const coords = f.geometry.coordinates[0].map(c => [c[1], c[0]]);
            const label = f.properties.label;
            counts[label] = (counts[label] || 0) + 1;
            const color = getClassColor(label);
            L.polygon(coords, {{ color: color, weight: 1.5, fillOpacity: 0.2 }})
                .bindTooltip(`${{label}} (${{f.properties.score.toFixed(2)}})`) 
                .addTo(detectionLayer);
        }});
        statusEl.textContent = `Detected ${{data.features.length}} object(s).`;
        updateLegend(counts);
    }} catch (err) {{
        statusEl.textContent = 'Request failed: ' + err.message;
    }}
}};

document.getElementById('btnDownload').onclick = () => {{
    if (!lastGeoJSON) {{ alert('No detection results to download!'); return; }}
    const blob = new Blob([JSON.stringify(lastGeoJSON, null, 2)], {{ type: 'application/json' }});
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = `detection_${{modelSelect.value}}.geojson`;
    a.click();
    URL.revokeObjectURL(url);
}};

document.getElementById('btnClear').onclick = () => {{
    detectionLayer.clearLayers();
    lastGeoJSON = null;
    updateLegend({{}});
    statusEl.textContent = 'Detections cleared.';
}};
</script>
</body>
</html>
"""
    return HTMLResponse(content=html)

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
await server.serve()
